# TUGAS KECERDASAN ARTIFISIAL UNTUK GEOFISIKA

## Perancangan Prompt AI untuk Pengolahan dan Interpretasi Data Geolistrik

### Menggunakan Struktur R–T–C–I–K–O–Q–V

#### **Nama:** **Aleksander**
#### **NIM:** *(*H061251003*)*
#### **Mata Kuliah:** Kecerdasan Artifisial untuk Geofisika 
#### **Departemen Geofisika** 
#### **Universitas Hasanuddin**

------------------------------------------------------------------------

## Struktur Prompt

Prompt disusun sebagai *work specification* (spesifikasi pekerjaan),
bukan sekadar pertanyaan. Setiap komponen dipakai untuk mengurangi
ambiguitas agar hasil AI dapat diperiksa, bukan hanya terlihat
meyakinkan.

| Komponen         | Simbol | Fungsi                                                                                                      |
|------------------------|------------------------|------------------------|
| Role             | **R**  | Sudut pandang kerja AI; menentukan istilah dan prioritas, bukan bukti kredensial                            |
| Task             | **T**  | Aksi utama dan hasil yang diminta, memakai kata kerja yang dapat diperiksa (hitung, bandingkan, inversikan) |
| Context          | **C**  | Tujuan penggunaan hasil dan batas tafsirnya                                                                 |
| Input            | **I**  | Data yang boleh dipakai, lengkap dengan format, satuan, dan konfigurasi elektroda                           |
| Constraints      | **K**  | Batasan yang dapat diuji pada prosedur atau hasil                                                           |
| Output           | **O**  | Bentuk, isi, dan nama berkas hasil                                                                          |
| Quality Criteria | **Q**  | Kriteria layak pakai yang terukur, bukan kata umum seperti “akurat”                                         |
| Verification     | **V**  | Pemeriksaan yang dapat dijalankan, beserta kondisi yang diharapkan dan tindakan bila gagal                  |

Bagian yang perlu diisi sebelum prompt dipakai ditulis dalam tanda
kurung siku, misalnya `[NAMA_FILE]`.

## Daftar Prompt

Kelima prompt mengikuti alur kerja data geolistrik dari awal sampai
akhir.

1.  Pemeriksaan Kualitas Data dan Perhitungan Resistivitas Semu
2.  Interpretasi Sounding Satu Dimensi (VES) Konfigurasi Schlumberger
3.  Inversi Dua Dimensi (ERT) dan Pembacaan Penampang Resistivitas
4.  Interpretasi Litologi dan Zona Air Tanah dengan Data Bor
5.  Analisis Ekuivalensi dan Ketidakpastian Model (Parameter Dar
    Zarrouk)

------------------------------------------------------------------------

# Prompt 1 — Pemeriksaan Kualitas Data dan Perhitungan Resistivitas Semu

``` markdown
# R — Role
Anda berperan sebagai ahli metode geolistrik yang membimbing mahasiswa S1 Geofisika dalam memeriksa kualitas data lapangan dan menghitung resistivitas semu.

# T — Task
Periksa kualitas data pengukuran geolistrik, hitung faktor geometri (K) dan resistivitas semu (ρa) untuk setiap titik ukur, lalu tandai data yang mencurigakan.

# C — Context
Pemeriksaan ini adalah tahap awal sebelum pemodelan atau inversi pada praktikum Metode Geolistrik. Hasil hanya dipakai untuk menilai kualitas dan kelayakan data. Interpretasi geologi bukan bagian dari tugas ini.

# I — Input
- Berkas [NAMA_FILE], format [CSV/XLSX/TXT], berisi posisi elektroda arus (A, B) dan potensial (M, N) dalam meter, arus I (mA), beda potensial ΔV (mV), dan bila ada pengukuran ulang atau resiprokal.
- Konfigurasi elektroda: [Wenner / Schlumberger / dipole-dipole / lainnya].
- Rumus (ρa dalam Ω·m, K dalam m, ΔV dalam V, I dalam A):
  - ρa = K · ΔV / I
  - Wenner: K = 2π·a
  - Schlumberger: K = π · ((AB/2)² − (MN/2)²) / MN
  - Dipole-dipole: K = π · n · (n+1) · (n+2) · a
- Gunakan hanya data ini.

# K — Constraints
- Jangan mengubah data asli. Konversi satuan (mA ke A, mV ke V) dilakukan pada kolom baru dan dijelaskan.
- Jangan menghapus data yang mencurigakan. Beri flag beserta alasannya.
- Jelaskan cara mendeteksi data buruk, misalnya ρa negatif, arus terlalu kecil, ΔV di bawah batas kepekaan alat, atau selisih pengukuran ulang yang besar. Nyatakan ambang yang dipakai.
- Jangan menghitung kedalaman investigasi sebagai kedalaman sebenarnya. Bila disebut, jelaskan sebagai perkiraan.
- Jangan membuat interpretasi litologi.

# O — Output
1. Berkas `resistivitas_semu.csv` dengan kolom: posisi elektroda, konfigurasi, I (A), ΔV (V), K (m), ρa (Ω·m), dan kolom `flag` beserta alasannya.
2. Ringkasan jumlah data total, jumlah data ter-flag, dan rentang nilai ρa (minimum, maksimum, median).
3. Gambar ρa terhadap jarak elektroda (atau pseudosection untuk data 2D) dengan data ter-flag diberi tanda berbeda.
4. Penjelasan singkat mengapa kualitas data penting sebelum inversi, dalam bentuk paragraf.
5. Kode Python beserta versi paket yang dipakai.

# Q — Quality Criteria
- Jumlah baris output sama dengan jumlah baris input.
- Semua satuan konsisten dan tercantum pada tiap kolom.
- Setiap flag memiliki alasan dan ambang yang dapat ditelusuri.
- Tidak ada nilai ρa yang tidak berhingga atau kosong tanpa penjelasan.
- Sumbu gambar berlabel satuan dan berskala logaritmik untuk ρa.

# V — Verification
- Hitung ulang satu titik ukur secara manual dan bandingkan dengan hasil program (selisih relatif < 1e-6).
- Uji rumus K dengan kasus sederhana: Wenner dengan a = 10 m harus memberi K = 62,83 m.
- Periksa bahwa jumlah data ter-flag ditambah data tidak ter-flag sama dengan jumlah data total.
- Bila ada pemeriksaan yang gagal, hentikan dan laporkan penyebabnya sebelum membuat gambar akhir.
```

------------------------------------------------------------------------

# Prompt 2 — Interpretasi Sounding Satu Dimensi (VES) Konfigurasi Schlumberger

``` markdown
# R — Role
Anda berperan sebagai ahli geolistrik dan pemodelan satu dimensi yang membimbing mahasiswa S1 Geofisika dalam menginterpretasikan kurva sounding.

# T — Task
Interpretasikan kurva ρa terhadap AB/2 dari data VES untuk menentukan jumlah lapisan, resistivitas sebenarnya (ρ), dan ketebalan (h) tiap lapisan, serta tentukan tipe kurvanya.

# C — Context
VES dipakai pada praktikum Metode Geolistrik untuk memperkirakan struktur berlapis bawah permukaan, misalnya lapisan penutup, lapisan pembawa air, dan batuan dasar. Hasil dipakai sebagai model perkiraan untuk laporan praktikum, bukan penentuan litologi pasti.

# I — Input
- Berkas [NAMA_FILE] berisi AB/2 (m), MN/2 (m), dan ρa (Ω·m) untuk [___] titik sounding.
- Perhatikan pergantian MN pada kurva dan penyambungan segmennya.
- Gunakan hanya data ini. Jangan memakai data lain kecuali data bor yang diberikan secara terpisah [ada/tidak ada].

# K — Constraints
- Jangan mengubah data asli. Bila segmen MN yang berbeda disambung atau digeser, jelaskan caranya.
- Jelaskan alasan pemilihan jumlah lapisan. Jangan menambah lapisan hanya untuk memperkecil galat. Bandingkan minimal dua model dengan jumlah lapisan berbeda.
- Nyatakan metode yang dipakai (pencocokan kurva, inversi kuadrat terkecil, atau lainnya), model awal, dan batas nilai parameter.
- Nyatakan tipe kurva (H, K, A, Q untuk tiga lapisan atau kombinasinya) dan dasar penentuannya.
- Jangan menyimpulkan jenis batuan hanya dari nilai resistivitas. Interpretasi litologi hanya boleh diajukan sebagai dugaan yang dijelaskan dasarnya.
- Jelaskan bahwa solusi inversi tidak unik.

# O — Output
1. Berkas `model_ves.csv`: nomor lapisan, ρ (Ω·m), h (m), kedalaman dasar lapisan (m), dan galat model.
2. Gambar kurva data dan kurva model pada sumbu log–log, serta penampang model berlapis.
3. Tabel perbandingan model dengan jumlah lapisan berbeda beserta galatnya (RMS error, %).
4. Interpretasi dalam bentuk paragraf yang memisahkan hasil hitungan dari dugaan geologi.
5. Keterbatasan hasil, kesimpulan singkat, dan kode Python.

# Q — Quality Criteria
- Galat model dinyatakan dengan angka (RMS error dalam %), bukan hanya "cocok".
- Jumlah lapisan mempunyai alasan yang jelas dan tidak berlebihan terhadap jumlah data.
- Semua satuan konsisten dan tiap parameter mempunyai nilai serta satuan.
- Interpretasi tidak melampaui apa yang didukung data dan dapat dipakai langsung dalam laporan praktikum.

# V — Verification
- Hitung kurva ρa maju (forward) dari model hasil inversi, lalu bandingkan dengan data. Galatnya harus sama dengan yang dilaporkan.
- Ubah model awal (resistivitas dan tebal awal) dan ulangi inversi. Bila hasilnya berubah besar, laporkan bahwa solusi bergantung pada model awal.
- Periksa bahwa kedalaman lapisan terdalam wajar terhadap AB/2 maksimum, dan tandai lapisan yang berada di luar jangkauan data.
- Bila ada pemeriksaan yang gagal, hentikan dan laporkan sebelum menyusun interpretasi.
```

------------------------------------------------------------------------

# Prompt 3 — Inversi Dua Dimensi (ERT) dan Pembacaan Penampang Resistivitas

``` markdown
# R — Role
Anda berperan sebagai ahli Electrical Resistivity Tomography (ERT) dan inversi geofisika yang membimbing mahasiswa S1 Geofisika dalam mengolah dan membaca penampang resistivitas 2D.

# T — Task
Lakukan inversi 2D data ERT untuk memperoleh penampang resistivitas sebenarnya, evaluasi kualitas inversi dengan galat RMS, lalu jelaskan pola resistivitas tinggi dan rendah pada penampang.

# C — Context
ERT dipakai pada praktikum Metode Geolistrik untuk melihat variasi resistivitas secara lateral dan vertikal, misalnya perubahan material atau zona jenuh air. Hasil dipakai untuk interpretasi awal dalam laporan praktikum, bukan keputusan teknis final.

# I — Input
- Berkas [NAMA_FILE] berisi posisi elektroda (m), ρa (Ω·m), konfigurasi [Wenner / Wenner-Schlumberger / dipole-dipole], spasi elektroda [___ m], dan jumlah elektroda [___].
- Data topografi lintasan [ada/tidak ada]. Bila ada, sebutkan format dan satuannya.
- Perangkat lunak atau pustaka inversi yang dipakai: [Res2DInv / pyGIMLi / lainnya].

# K — Constraints
- Jangan mengubah data asli. Data buruk yang dikeluarkan sebelum inversi harus dicatat jumlah, posisi, dan alasannya.
- Nyatakan parameter inversi: jenis regularisasi (misalnya smoothness-constrained L2 atau robust L1), faktor redaman, model awal, dan kriteria berhenti.
- Jangan menghentikan iterasi hanya karena RMS kecil. Jelaskan kewajarannya terhadap kualitas data dan jumlah iterasi.
- Sertakan topografi bila datanya ada. Bila tidak ada, nyatakan asumsi permukaan datar.
- Jangan menafsirkan zona di bagian tepi dan dasar penampang yang tidak dicakup data seolah-olah sama andalnya.
- Jangan menyimpulkan litologi hanya dari warna atau nilai resistivitas.

# O — Output
1. Penampang hasil inversi (jarak terhadap kedalaman) dengan skala warna logaritmik, satuan Ω·m, dan sumbu berlabel.
2. Tabel RMS error tiap iterasi dan RMS akhir (%).
3. Perbandingan tiga penampang: data ρa terukur, ρa hasil perhitungan model, dan model resistivitas sebenarnya.
4. Deskripsi pola resistivitas (zona tinggi, rendah, dan batasnya) dalam bentuk paragraf, lalu dugaan geologi yang dijelaskan dasarnya.
5. Keterbatasan hasil dan kode atau langkah kerja perangkat lunak yang dipakai.

# Q — Quality Criteria
- RMS akhir dilaporkan dengan angka dan dinilai terhadap kualitas data, bukan hanya "bagus".
- Skala warna, kedalaman, dan sumbu tercantum dan tidak dipotong tanpa penanda.
- Deskripsi pola dipisahkan dari dugaan geologi.
- Penjelasan dapat diikuti mahasiswa S1 dan dapat dipakai langsung dalam laporan.

# V — Verification
- Bandingkan penampang ρa terukur dengan ρa hasil perhitungan model. Pola umumnya harus mirip dan selisih besar harus dijelaskan.
- Ulangi inversi dengan faktor redaman berbeda. Bila struktur utama berubah besar, laporkan bahwa struktur tersebut tidak stabil.
- Periksa bahwa perubahan RMS antar iterasi melandai. Bila terus turun tajam, laporkan bahwa inversi belum konvergen.
- Bila ada pemeriksaan yang gagal, hentikan dan laporkan sebelum menyusun interpretasi.
```

------------------------------------------------------------------------

# Prompt 4 — Interpretasi Litologi dan Zona Air Tanah dengan Data Bor

``` markdown
# R — Role
Anda berperan sebagai ahli geofisika dan hidrogeologi yang membimbing mahasiswa S1 Geofisika dalam menghubungkan nilai resistivitas dengan litologi dan keberadaan air tanah.

# T — Task
Bandingkan nilai resistivitas hasil pemodelan dengan data bor, kelompokkan lapisan ke dalam kelas resistivitas, lalu tentukan lapisan yang berpotensi menjadi akuifer beserta tingkat keyakinannya.

# C — Context
Analisis dilakukan pada praktikum Metode Geolistrik dan Hidrogeofisika. Nilai resistivitas dipengaruhi oleh litologi, porositas, kandungan lempung, dan kualitas air, sehingga satu nilai resistivitas dapat berasal dari lebih dari satu kemungkinan. Hasil dipakai sebagai interpretasi awal, bukan penentuan potensi air tanah final.

# I — Input
- Model resistivitas hasil VES atau ERT: [NAMA_FILE] berisi kedalaman (m) dan ρ (Ω·m).
- Data bor atau sumur terdekat: [NAMA_FILE] berisi kedalaman (m), deskripsi litologi, dan muka air tanah (m) bila ada. Jarak titik bor dari lintasan ukur [___ m].
- Tabel rentang resistivitas acuan: [sumber acuan yang dipakai].
- Gunakan hanya data ini.

# K — Constraints
- Jangan mengubah nilai resistivitas hasil pemodelan.
- Sebutkan sumber tabel rentang resistivitas. Jangan memakai satu rentang universal karena rentang antarjenis material saling tumpang tindih.
- Jelaskan ketidakpastian akibat perbedaan skala (resolusi geolistrik terhadap deskripsi bor) dan jarak titik bor dari lintasan.
- Jangan menyimpulkan akuifer produktif hanya dari nilai resistivitas. Lempung jenuh air asin dan batuan kompak sama-sama dapat menghasilkan resistivitas rendah atau tinggi yang menyesatkan.
- Bila relasi Archie (ρ = a · ρw · φ^−m · Sw^−n) dipakai, nyatakan semua parameternya dan sumbernya. Jangan menebak nilai porositas.
- Jangan membuat perkiraan debit atau kualitas air.

# O — Output
1. Tabel korelasi kedalaman: lapisan resistivitas, litologi bor, dan tingkat kecocokan (sesuai / sebagian / tidak sesuai) beserta alasannya.
2. Tabel kelas resistivitas dengan dugaan material dan tingkat keyakinan (tinggi / sedang / rendah).
3. Gambar model resistivitas berdampingan dengan kolom litologi bor pada skala kedalaman yang sama.
4. Interpretasi dalam bentuk paragraf tentang lapisan yang berpotensi menjadi akuifer, disertai alternatif tafsir lain.
5. Keterbatasan dan kesimpulan singkat.

# Q — Quality Criteria
- Setiap dugaan litologi mempunyai dasar yang dapat ditelusuri ke data bor atau ke tabel acuan yang disebut.
- Tingkat keyakinan dinyatakan eksplisit dan tidak seragam untuk semua lapisan.
- Alternatif tafsir dicantumkan untuk lapisan yang ambigu.
- Kedalaman dan satuan konsisten pada seluruh tabel dan gambar.

# V — Verification
- Periksa bahwa kedalaman batas lapisan geolistrik dan batas litologi bor dibandingkan pada satuan dan datum yang sama.
- Periksa apakah kedalaman muka air tanah dari data bor konsisten dengan penurunan resistivitas pada model. Bila tidak, jelaskan kemungkinan penyebabnya.
- Untuk tiap lapisan yang dinilai berpotensi akuifer, sebutkan minimal satu bukti yang dapat menyangkalnya, lalu nilai apakah bukti itu terbantahkan oleh data.
- Bila ada pemeriksaan yang gagal, turunkan tingkat keyakinan lapisan tersebut dan laporkan alasannya.
```

------------------------------------------------------------------------

# Prompt 5 — Analisis Ekuivalensi dan Ketidakpastian Model (Parameter Dar Zarrouk)

``` markdown
# R — Role
Anda berperan sebagai peninjau metode inversi geolistrik yang membimbing mahasiswa S1 Geofisika dalam menilai seberapa andal model hasil interpretasi.

# T — Task
Hitung parameter Dar Zarrouk (konduktansi longitudinal S dan resistansi transversal T), lalu uji prinsip ekuivalensi untuk menunjukkan rentang model yang sama-sama cocok dengan data VES.

# C — Context
Pada VES, beberapa kombinasi ketebalan dan resistivitas dapat menghasilkan kurva ρa yang hampir sama (prinsip ekuivalensi). Analisis dilakukan pada praktikum Metode Geolistrik untuk memahami bahwa hasil pemodelan mempunyai ketidakpastian. Hasil dipakai untuk menyatakan rentang parameter yang wajar, bukan nilai tunggal.

# I — Input
- Model VES hasil interpretasi [NAMA_FILE]: ρ (Ω·m) dan h (m) tiap lapisan.
- Data VES asli [NAMA_FILE]: AB/2 (m) dan ρa (Ω·m).
- Batas toleransi galat yang dianggap masih cocok dengan data: [misalnya RMS error ≤ ___ %].
- Rumus:
  - S = Σ (hᵢ / ρᵢ), dalam siemens (S)
  - T = Σ (hᵢ · ρᵢ), dalam Ω·m²
- Gunakan hanya data ini.

# K — Constraints
- Jangan mengubah model dan data asli. Semua variasi model dibuat pada salinan.
- Nyatakan lapisan yang diuji ekuivalensinya dan alasannya: lapisan konduktif diuji terhadap S konstan, lapisan resistif diuji terhadap T konstan.
- Nyatakan cara membuat variasi model (rentang perubahan ρ dan h, jumlah model yang diuji) dan kriteria diterimanya sebuah model.
- Jangan menyatakan satu model sebagai "benar". Laporkan rentang parameter yang memenuhi toleransi.
- Jelaskan keterbatasan bila jumlah data kecil atau sebaran AB/2 sempit.

# O — Output
1. Tabel S dan T tiap lapisan dan total, beserta satuannya.
2. Berkas `model_ekuivalen.csv` berisi model-model yang memenuhi toleransi: ρ, h, S atau T, dan RMS error.
3. Gambar: (a) kurva data dan beberapa kurva model ekuivalen, (b) rentang ρ dan h yang diterima untuk lapisan yang diuji.
4. Interpretasi dalam bentuk paragraf tentang parameter yang tertentu dengan baik dan yang tidak, serta dampaknya pada interpretasi geologi.
5. Keterbatasan, kesimpulan singkat, dan kode Python.

# Q — Quality Criteria
- Rentang parameter dilaporkan dengan angka dan satuan, tidak hanya nilai tunggal.
- Setiap model ekuivalen memenuhi toleransi galat yang dinyatakan.
- Penjelasan ekuivalensi benar secara fisis dan mudah dipahami mahasiswa S1.
- Kesimpulan menyatakan parameter mana yang andal dan mana yang tidak.

# V — Verification
- Periksa bahwa S (untuk uji lapisan konduktif) atau T (untuk uji lapisan resistif) pada model ekuivalen tetap sama dengan model awal (selisih relatif < 1%).
- Hitung kurva ρa maju dari tiap model ekuivalen dan pastikan RMS error-nya sesuai yang dilaporkan.
- Bandingkan rentang yang diperoleh dengan hasil dua model dengan jumlah lapisan berbeda dari Prompt 2. Laporkan bila ada perbedaan besar.
- Bila ada pemeriksaan yang gagal, hentikan dan laporkan sebelum membuat kesimpulan.
```

------------------------------------------------------------------------

## Catatan Penyusunan

-   Kelima prompt mengikuti alur data geolistrik: pemeriksaan data,
    pemodelan 1D, inversi 2D, korelasi dengan data bor, lalu penilaian
    ketidakpastian.
-   Seluruh prompt memisahkan hasil hitungan dari interpretasi dan
    membatasi klaim sesuai data yang tersedia.
-   Setiap prompt memuat pemeriksaan yang dapat dijalankan beserta
    tindakan bila pemeriksaan gagal.
-   Bagian dalam tanda kurung siku `[ ]` diisi sesuai data geolistrik
    praktikum sebelum prompt dikirim ke AI.